### Installing Neccessary Libraries

In [1]:
!pip install colbert-ai[faiss-gpu,torch]
!pip install python-terrier
!pip install ir-datasets
!pip install numpy pandas scipy scikit-learn
!pip install transformers
!pip install sentence-transformers
!pip install ranx 
!pip install pytrec_eval
!pip install tqdm
!pip install datasets  
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.4/887.4 MB 293.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 295.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 297.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 653.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 296.1 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: nvidia-cuda-runtime-cu11
    Found existing installation: nvidia-cuda-runtime-cu11 11.8.89
    Uninstalling nvidia-cuda-runtime-cu11-11.8.89:
      Successfully uninstalled nvidia-cuda-runtime-cu11-11.8.890m [nvidia-cuda-runtime-cu11]
  Attempting uninstall: nvidia-cuda-nvrtc-cu11━━ 0/5 [nvidia-cuda-runtime-cu11]
    Found existing installation: nvidia-cuda-nvrtc-cu11 11.8.89nvidia-cuda-runtime-cu11]
    Uninstalling nvidia-cuda-nvrtc-cu11-11.8.89: 0/5 [nvidia-cuda-runtime-cu11]
      Successfully uninstalled nvidia-cuda-nvrtc-cu11-11.8.8

### Importing neccessary libraries

In [2]:
import pandas as pd
import pyterrier as pt
import ir_datasets
from colbert import Searcher
from colbert.infra import Run, RunConfig
import os

/opt/miniconda3/envs/colbertv2/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Start PyTerrier only if it is not already started
if not pt.started():
    pt.init()

/tmp/ipykernel_450/3174203117.py:2: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_450/3174203117.py:3: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


### Load SARA Dataset

In [4]:
# Load the SARA dataset from ir_datasets
dataset = ir_datasets.load('sara')

# Extract queries and relevance judgments into lists
topics_list = list(dataset.queries_iter())
qrels_list = list(dataset.qrels_iter())

# Collect all document texts
docs = [doc.text for doc in dataset.docs_iter()]

# Convert queries to a PyTerrier-compatible DataFrame
topics = pd.DataFrame([
    {'qid': topic.query_id, 'query': topic.text} 
    for topic in topics_list
])

# Convert qrels (relevance judgments) to a PyTerrier-compatible DataFrame
qrels = pd.DataFrame([
    {'qid': qrel.query_id, 'docno': str(qrel.doc_id), 'label': int(qrel.relevance)}
    for qrel in qrels_list
])

# Print dataset statistics
print(f"No. of Queries: {len(topics)},  No. of Documents: {len(docs)}, Total Qrels: {len(qrels)}")

No. of Queries: 150,  No. of Documents: 1702, Total Qrels: 34413


### Setting Up Document ID Mapping

In [5]:
# Set index name and path for the document ID mapping file
index_name = 'colbert_sara_index'
mapping_path = f"./Index/{index_name}_doc_mapping.txt"

# Display mapping file location
print(f"Document ID mapping from: {mapping_path} loaded ")

# Check if mapping file exists, otherwise raise error
if not os.path.exists(mapping_path):
    raise FileNotFoundError(f"Mapping file not found: {mapping_path}")

# Load mapping between ColBERT index IDs and SARA document IDs
idx_to_doc_id = {}
with open(mapping_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            idx, doc_id = parts
            idx_to_doc_id[int(idx)] = doc_id

# Print summary of loaded mappings
print(f"Mapping for {len(idx_to_doc_id)} documents loaded \n")
print(f"Sample mappings: {dict(list(idx_to_doc_id.items())[:3])}")

Document ID mapping from: ./Index/colbert_sara_index_doc_mapping.txt loaded 
Mapping for 1702 documents loaded 

Sample mappings: {0: '114715', 1: '229405', 2: '232795'}


### Load ColBERT Model

In [6]:
# Load the ColBERT index
print("Loading ColBERT searcher...")

# Use PyTerrier's Run context to set up the experiment
with Run().context(RunConfig(experiment='sara_experiment')):
    
    # Initialize the ColBERT searcher with the given index and model checkpoint
    searcher = Searcher(index=index_name, checkpoint="colbert-ir/colbertv2.0")
    print("Searcher loaded successfully")

Loading ColBERT searcher...
[Sep 08, 14:25:20] #> Loading codec...
[Sep 08, 14:25:21] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/opt/miniconda3/envs/colbertv2/lib/python3.9/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/opt/miniconda3/envs/colbertv2/lib/python3.9/site-packages/torch/utils/cpp_extension.py:2356: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(


[Sep 08, 14:26:59] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/opt/miniconda3/envs/colbertv2/lib/python3.9/site-packages/torch/utils/cpp_extension.py:2356: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(


[Sep 08, 14:28:22] #> Loading IVF...
[Sep 08, 14:28:22] #> Loading doclens...


100%|██████████| 1/1 [00:00<00:00, 58.61it/s]

[Sep 08, 14:28:22] #> Loading codes and residuals...



100%|██████████| 1/1 [00:00<00:00,  9.07it/s]

Searcher loaded successfully


### ColBERT Retrieval Function

In [7]:
def retrieve_colbert(topics, k=100):
    
    # Retrieve top-k documents for each query using ColBERT
    results = []
    
    print(f"Running retrieval for {len(topics)} queries...")
    for i, (_, row) in enumerate(topics.iterrows()):
        # Progress update every 10 queries
        if i % 10 == 0:
            print(f"Processing query {i+1}/{len(topics)}")
            
        qid = row['qid']
        query = row['query']
        
        try:
            # Search with ColBERT
            pids, ranks, scores = searcher.search(query, k=k)
            
            # Map ColBERT passage IDs to original SARA document IDs
            for rank, (pid, score) in enumerate(zip(pids, scores)):
                if pid in idx_to_doc_id:
                    original_doc_id = idx_to_doc_id[pid]
                    results.append({
                        'qid': qid,                       # query ID
                        'docno': str(original_doc_id),    # SARA document ID
                        'score': float(score),            # relevance score
                        'rank': rank                      # rank position
                    })
                else:
                    print(f"Warning: ColBERT PID {pid} not found in mapping")
                    
        except Exception as e:
            
            # Handle errors
            print(f"Error processing query {qid}: {e}")
            continue
    
    # Return results
    return pd.DataFrame(results)

In [8]:
# Run ColBERT retrieval for all queries
print("Running ColBERT retrieval")
scores = retrieve_colbert(topics, k=100)
print(f"Retrieved {len(scores)} results")

# Ensure results are not empty
if len(scores) == 0:
    print("ERROR: No results retrieved! Check your index and mapping.")
    exit(1)

Running ColBERT retrieval
Running retrieval for 150 queries...
Processing query 1/150


/opt/miniconda3/envs/colbertv2/lib/python3.9/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: Politicians that decide the plans from state to state, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([ 101,    1, 8801, 2008, 5630, 1996, 3488, 2013, 2110, 2000, 2110,  102,
         103,  103,  103,  103,  103,  103,  103,  103,  103,  103,  103,  103,
         103,  103,  103,  103,  103,  103,  103,  103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

Processing query 11/150
Processing query 21/150
Processing query 31/150
Processing query 41/150
Processing query 51/150
Processing query 61/150
Processing query 71/150
Processing query 81/150
Processing query 91/150
Processing query 101/150
Processing query 111/150
Processing query 121/150
Processing query 131/150
Processing query 141/150
Retrieved 15000 results


In [9]:
# Show the shape of the results DataFrame
print(scores.shape)
scores.head()

(15000, 4)


,qid,docno,score,rank
0,1,174414,11.296875,0
1,1,173952,11.101562,1
2,1,123470,11.070312,2
3,1,120514,10.804688,3
4,1,177810,10.765625,4


In [ ]:
# Save the results
scores.to_csv("./retrieval_scores/colbert_scores.csv", index=False)

### Evaluation

In [14]:
# Ensure doc IDs are strings in both results and qrels
scores['docno'] = scores['docno'].astype(str)
qrels['docno'] = qrels['docno'].astype(str)

# Run evaluation with PyTerrier
print("\nRunning evaluation...")

# Define evaluation metrics
metrics = ["ndcg_cut_10", "map", "P_10", "recall_10"]

# Evaluate retrieval results against qrels
eval_results = pt.Evaluate(scores, qrels, metrics=metrics)


Running evaluation...


In [15]:
# Convert evaluation results to a DataFrame
df = pd.DataFrame(eval_results, index=[0]).round(4)

# Add a column to label the model used
df.insert(0, "Model", "ColBERT")

In [16]:
df

,Model,ndcg_cut_10,map,P_10,recall_10
0,ColBERT,0.1167,0.073,0.11,0.1011
